# Análisis Detallado de un Fondo — Todos los Proveedores

Este notebook analiza un fondo concreto consultando **cada proveedor** de datos de forma independiente para comparar la cobertura y calidad de la información.

## Arquitectura de adquisición de datos (dual-strategy)

| Estrategia | Prioridad | Cadena de proveedores |
|---|---|---|
| **NAV / Histórico** | ⚡ Velocidad | FMP → YFinance → MorningStar(light) |
| **Info / Sectores / Holdings** | 📊 Completitud | MorningStar(detailed) → FMP → Finect |

### Proveedores disponibles

| Proveedor | NAV | Info | Sectores | Países | Holdings | Comisiones |
|-----------|-----|------|----------|--------|----------|------------|
| **FMP** | ✅ ~300ms | ETFs | ✅ | ✅ | ✅ | ❌ |
| **Morningstar** | ✅ ~2-5s | Muy rica | ✅ | ✅ | ✅ | Parcial |
| **Yahoo Finance** | ✅ ~500ms | Mínima | ❌ | ❌ | ❌ | ❌ |
| **Finect** | ❌ | Rica | ❌* | ❌ | ❌* | ✅ Detallado |

*Finect tiene holdings y asset allocation pero se renderizan con JavaScript (no disponibles en scraping estático).

### Lógica de fusión
- **NAV**: primer proveedor que devuelve precio > 0 gana (failover)
- **Info**: se fusionan TODOS los proveedores (primer valor no-nulo de cada campo gana)
- **Sectores/Países/Holdings**: se usa la fuente con más datos (mayor granularidad)

## 0. Setup e imports

In [1]:
import sys
import os
import importlib
import logging
import pandas as pd
from IPython.display import display, Markdown

# Resolver rutas — funciona tanto desde backend/notebooks/ como desde la raíz
if os.path.exists("../app"):
    sys.path.insert(0, os.path.abspath(".."))
    cache_path = "../data/cache"
else:
    sys.path.insert(0, os.path.abspath("./backend"))
    cache_path = "./backend/data/cache"

# Recargar módulos para recoger cambios en el código
import app.services.functions_fund
import app.services.data_providers
import app.services.finect_provider
import app.client
importlib.reload(app.services.functions_fund)
importlib.reload(app.services.data_providers)
importlib.reload(app.services.finect_provider)
importlib.reload(app.client)

# Configurar logging para ver qué hace cada proveedor
logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)
logger = logging.getLogger("notebook")

# Fondo a analizar — cambiar aquí para usar otro ISIN
ISIN = "IE00BYX5NX33"  # Vanguard FTSE All-World UCITS ETF (Acc)

# Defaults seguros para variables globales (evitan errores si alguna celda falla)
nav_fmp = nav_ms = nav_yf = nav_comp = None
info_fmp = info_ms = info_yf = info_finect = info_comp = {}
sectors_fmp = sectors_ms = sectors_comp = {}
countries_fmp = regions_ms = countries_comp = {}
alloc_finect = {}
holdings_fmp = holdings_ms = holdings_finect = holdings_comp = pd.DataFrame()
hist_fmp = hist_ms = hist_yf = pd.DataFrame()

print(f"🔍 Analizando fondo: {ISIN}")

🔍 Analizando fondo: IE00BYX5NX33


## 0.1 Diagnóstico y limpieza de caché

Comprueba si hay entradas de caché corruptas (con errores HTTP 403 u otros) y las elimina para permitir reintentos.

In [2]:
import pickle
import glob
from pathlib import Path
from datetime import datetime

cache_base = Path(cache_path).resolve()
isin_cache_dir = cache_base / ISIN

print(f"📁 Directorio de caché: {cache_base}")
print(f"📁 Caché del ISIN {ISIN}: {isin_cache_dir}\n")

corrupted = []
valid = []

if isin_cache_dir.exists():
    for pkl_file in sorted(isin_cache_dir.glob("*.pkl")):
        try:
            with open(pkl_file, "rb") as f:
                data = pickle.load(f)
            # Comprobar si es un DataFrame con columna error
            has_error = False
            if isinstance(data, pd.DataFrame):
                if "error" in data.columns and data["error"].notna().any():
                    has_error = True
                # También comprobar dentro de la columna 'data' (nested)
                if "data" in data.columns:
                    inner = data["data"].iloc[0]
                    if isinstance(inner, pd.DataFrame) and "error" in inner.columns:
                        if inner["error"].notna().any():
                            has_error = True
            
            age = datetime.now() - datetime.fromtimestamp(pkl_file.stat().st_mtime)
            status = "❌ ERROR" if has_error else "✅ OK"
            size_kb = pkl_file.stat().st_size / 1024
            
            if has_error:
                corrupted.append(pkl_file)
            else:
                valid.append(pkl_file)
            
            print(f"  {status} | {pkl_file.name} | {size_kb:.1f} KB | {age.days}d ago")
            if has_error and isinstance(data, pd.DataFrame):
                # Mostrar el error
                if "error" in data.columns:
                    print(f"         Error: {str(data['error'].iloc[0])[:120]}...")
                if "data" in data.columns:
                    inner = data["data"].iloc[0]
                    if isinstance(inner, pd.DataFrame) and "error" in inner.columns:
                        print(f"         Error (inner): {str(inner['error'].iloc[0])[:120]}...")
        except Exception as e:
            print(f"  ⚠️ No se pudo leer: {pkl_file.name} — {e}")
            corrupted.append(pkl_file)

    print(f"\n📊 Resumen: {len(valid)} válidos, {len(corrupted)} corruptos")
    
    if corrupted:
        print(f"\n🗑️ Eliminando {len(corrupted)} entradas corruptas...")
        for f in corrupted:
            f.unlink()
            print(f"   Eliminado: {f.name}")
        print("✅ Cache limpio. Las siguientes celdas harán peticiones frescas.")
    else:
        print("✅ No se encontraron entradas corruptas.")
else:
    print(f"ℹ️ No hay directorio de caché para {ISIN}")

📁 Directorio de caché: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
📁 Caché del ISIN IE00BYX5NX33: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache\IE00BYX5NX33

  ❌ ERROR | 08a21041e796b4b5aae7a57841799316.pkl | 147.0 KB | 0d ago
         Error (inner): Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidde...
  ✅ OK | 122c872adebe96552ebd90d2e0caae53.pkl | 145.8 KB | 0d ago
  ❌ ERROR | 50cc50bcd2ddb1207512368a83df4b39.pkl | 147.0 KB | 0d ago
         Error (inner): Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidde...
  ✅ OK | 54da87827db01f2509444632298146e1.pkl | 145.8 KB | 0d ago
  ✅ OK | a25f6cfc50164fcfc5fcbc144e979227.pkl | 146.6 KB | 0d ago
  ✅ OK | bca3d3c0427ece1697ef4b73ab98c8ec.pkl | 146.6 KB | 0d ago

📊 Resumen: 4 válidos, 2 corruptos

🗑️ Eliminando 2 

## 1. Proveedor FMP (FinancialModelingPrep)

Requiere API key (gratis, 250 req/día). Buena cobertura de ETFs internacionales.

In [3]:
from app.services.data_providers import FMPProvider

fmp = FMPProvider()
print(f"FMP disponible: {fmp.available}")

if fmp.available:
    # 1a. Resolver ISIN → ticker symbol
    symbol = fmp.resolve_symbol(ISIN)
    print(f"Symbol resuelto: {symbol}\n")

    # 1b. NAV actual
    nav_fmp = fmp.get_nav(ISIN)
    print(f"NAV actual: {nav_fmp}")

    # 1c. Info del fondo
    info_fmp = fmp.get_fund_info(ISIN)
    print("\n--- Info FMP ---")
    display(pd.DataFrame(list(info_fmp.items()), columns=["Campo", "Valor"]))

    # 1d. Distribución sectorial
    sectors_fmp = fmp.get_sector_weights(ISIN)
    print("\n--- Sectores FMP ---")
    if sectors_fmp:
        display(pd.DataFrame(list(sectors_fmp.items()), columns=["Sector", "Peso (%)"]))
    else:
        print("(sin datos)")

    # 1e. Distribución geográfica
    countries_fmp = fmp.get_country_weights(ISIN)
    print("\n--- Países FMP ---")
    if countries_fmp:
        display(pd.DataFrame(list(countries_fmp.items()), columns=["País", "Peso (%)"]))
    else:
        print("(sin datos)")

    # 1f. Holdings
    holdings_fmp = fmp.get_holdings(ISIN)
    print(f"\n--- Holdings FMP ({len(holdings_fmp)} posiciones) ---")
    display(holdings_fmp.head(10))

    # 1g. Histórico de precios (últimos 3 años)
    hist_fmp = fmp.get_nav_history(ISIN, years=3)
    print(f"\n--- Histórico FMP: {len(hist_fmp)} registros ---")
    if not hist_fmp.empty:
        display(hist_fmp.describe())
else:
    print("⚠️ FMP no disponible (sin API key). Saltando.")

FMP disponible: True
Symbol resuelto: None

NAV actual: None

--- Info FMP ---


,Campo,Valor



--- Sectores FMP ---
(sin datos)

--- Países FMP ---
(sin datos)

--- Holdings FMP (0 posiciones) ---


,name,ticker,weight,market_value



--- Histórico FMP: 0 registros ---


## 2. Proveedor Morningstar (mstarpy)

Fuente más rica en datos cualitativos: ratings, analyst driven, costes, distribución regional/sectorial, risk score.
Usa la clase `Fund` en modo `detailed` para obtener toda la información.

In [4]:
from app.services.data_providers import MStarProvider

mstar = MStarProvider(cache_path=cache_path)

# 2a. NAV actual
nav_ms = mstar.get_nav(ISIN)
print(f"NAV Morningstar: {nav_ms}")

# 2b. Info del fondo (modo detallado)
info_ms = mstar.get_fund_info(ISIN)
print("\n--- Info Morningstar ---")
display(pd.DataFrame(list(info_ms.items()), columns=["Campo", "Valor"]))

# 2c. Distribución sectorial
sectors_ms = mstar.get_sector_weights(ISIN)
print("\n--- Sectores Morningstar ---")
if sectors_ms:
    display(pd.DataFrame(list(sectors_ms.items()), columns=["Sector", "Peso (%)"]))
else:
    print("(sin datos)")

# 2d. Distribución regional
regions_ms = mstar.get_country_weights(ISIN)
print("\n--- Regiones Morningstar ---")
if regions_ms:
    display(pd.DataFrame(list(regions_ms.items()), columns=["Región", "Peso (%)"]))
else:
    print("(sin datos)")

# 2e. Holdings
holdings_ms = mstar.get_holdings(ISIN)
print(f"\n--- Holdings Morningstar ({len(holdings_ms)} posiciones) ---")
display(holdings_ms.head(10))

# 2f. Histórico de precios
hist_ms = mstar.get_nav_history(ISIN, years=3)
print(f"\n--- Histórico Morningstar: {len(hist_ms)} registros ---")
if not hist_ms.empty:
    display(hist_ms.describe())

Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BYX5NX33 (light)
NAV Morningstar: 13.045499801635742
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5NX33
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...


app.services.data_providers — MStarProvider: cache de IE00BYX5NX33 (detailed) contiene errores, reintentando sin cache


❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos para IE00BYX5NX33: 2053 días
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...
❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)

--- Info Morningstar ---


,Campo,Valor
0,isin,IE00BYX5NX33
1,source,Morningstar
2,name,IE00BYX5NX33


Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5NX33
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...


...

app.services.data_providers — MStarProvider: cache de IE00BYX5NX33 (detailed) contiene errores, reintentando sin cache


❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos para IE00BYX5NX33: 2053 días
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...
❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)

--- Holdings Morningstar (0 posiciones) ---


,name,ticker,weight,market_value


Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BYX5NX33 (light)

--- Histórico Morningstar: 759 registros ---


,date,price
count,759,759.000000
mean,2024-10-24 14:46:00,10.649227
min,2023-05-02 00:00:00,8.013700
25%,2024-01-25 12:00:00,9.471100
50%,2024-10-22 00:00:00,10.737000
75%,2025-07-24 12:00:00,11.811150
max,2026-04-27 00:00:00,13.062000
std,NaN,1.386214


### 2 bis. Datos crudos de Morningstar (clase Fund directa)

Accedemos directamente a la clase `Fund` para inspeccionar el DataFrame interno con **todos** los campos que devuelve mstarpy.

In [14]:
from app.services.functions_fund import Fund

# Forzar use_cache=True: si la caché se limpió en la celda anterior, hará peticiones frescas
fund_obj = Fund(isin=ISIN, mode="detailed", cache_path=cache_path, use_cache=True)
df_raw = fund_obj.fund_data

print(f"Columnas del DataFrame raíz: {list(df_raw.columns)}")
print(f"Nombre del fondo: {df_raw['nombre'].iloc[0]}")

# Inspeccionar la columna 'data' que contiene toda la info de Morningstar
data_col = df_raw["data"].iloc[0]

# Detectar si los datos contienen error
if isinstance(data_col, pd.DataFrame) and "error" in data_col.columns and data_col["error"].notna().any():
    error_msg = str(data_col["error"].iloc[0])
    print(f"\n⚠️ Los datos de Morningstar contienen un ERROR:")
    print(f"   {error_msg[:300]}")
    print(f"\n   Esto significa que la API de Morningstar (mstarpy) devolvió un error.")
    print(f"   Causa probable: HTTP 403 Forbidden — Morningstar bloquea temporalmente el acceso.")
    print(f"   Solución: esperar unas horas y reintentar, o usar un proxy/VPN.")
    print(f"\n   Campos disponibles a pesar del error: {list(data_col.columns)}")
elif isinstance(data_col, pd.DataFrame):
    print(f"\n--- Columnas en 'data' ({len(data_col.columns)} campos) ---")
    # Mostrar todos los campos con su valor
    row = data_col.iloc[0]
    campos = pd.DataFrame({
        "Campo": row.index,
        "Valor": row.values,
        "Tipo": [type(v).__name__ for v in row.values],
    })
    # Filtrar campos no nulos para mejor lectura
    campos_validos = campos[campos["Valor"].apply(lambda x: x is not None and str(x) != "nan")]
    print(f"Campos con datos: {len(campos_validos)} de {len(campos)}")
    display(campos_validos.reset_index(drop=True))
elif isinstance(data_col, dict):
    display(pd.DataFrame(list(data_col.items()), columns=["Campo", "Valor"]))
else:
    print(f"Tipo inesperado: {type(data_col)}")

# Inspeccionar histórico
hist_col = df_raw["historical_data"].iloc[0]
if isinstance(hist_col, pd.DataFrame) and not hist_col.empty:
    print(f"\n--- Histórico: {len(hist_col)} registros, columnas: {list(hist_col.columns)} ---")
    display(hist_col.tail(5))
else:
    print(f"\n⚠️ Sin datos históricos")

Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5NX33
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...
❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)
Columnas del DataFrame raíz: ['isin', 'nombre', 'historical_data', 'data']
Nombre del fondo: IE00BYX5NX33

⚠️ Los datos de Morningstar contienen un ERROR:
   Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.

   Esto significa que la API de Morningstar (mstarpy) devolvió un error.
   Causa probable: HTTP 403 Forbidden — Morningstar bloquea temporalmente el acceso.


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,
2026-04-21 00:00:00+02:00,12.9224,12.9224,12.9224,12.9224,0,0.0,0.0,0.0
2026-04-22 00:00:00+02:00,13.0364,13.0364,13.0364,13.0364,0,0.0,0.0,0.0
2026-04-23 00:00:00+02:00,12.9993,12.9993,12.9993,12.9993,0,0.0,0.0,0.0
2026-04-24 00:00:00+02:00,13.0620,13.0620,13.0620,13.0620,0,0.0,0.0,0.0
2026-04-27 00:00:00+02:00,13.0455,13.0455,13.0455,13.0455,0,0.0,0.0,0.0


## 3. Proveedor Yahoo Finance

Proveedor ligero de respaldo. Solo aporta NAV e histórico de precios.

In [6]:
from app.services.data_providers import YFinanceProvider

yf_prov = YFinanceProvider()

# 3a. NAV actual
nav_yf = yf_prov.get_nav(ISIN)
print(f"NAV Yahoo Finance: {nav_yf}")

# 3b. Info básica
info_yf = yf_prov.get_fund_info(ISIN)
print("\n--- Info Yahoo Finance ---")
display(pd.DataFrame(list(info_yf.items()), columns=["Campo", "Valor"]))

# 3c. Histórico de precios
hist_yf = yf_prov.get_nav_history(ISIN, years=3)
print(f"\n--- Histórico Yahoo Finance: {len(hist_yf)} registros ---")
if not hist_yf.empty:
    display(hist_yf.describe())
    print(f"Rango: {hist_yf['date'].min()} → {hist_yf['date'].max()}")

NAV Yahoo Finance: 13.045499801635742

--- Info Yahoo Finance ---


,Campo,Valor
0,name,Fidelity MSCI World Index EUR P Acc
1,currency,EUR
2,source,YahooFinance



--- Histórico Yahoo Finance: 759 registros ---


,date,price
count,759,759.000000
mean,2024-10-24 14:46:00,10.649227
min,2023-05-02 00:00:00,8.013700
25%,2024-01-25 12:00:00,9.471100
50%,2024-10-22 00:00:00,10.737000
75%,2025-07-24 12:00:00,11.811150
max,2026-04-27 00:00:00,13.062000
std,NaN,1.386214


Rango: 2023-05-02 00:00:00 → 2026-04-27 00:00:00


## 4. Proveedor Finect (scraping)

Scraping del HTML público de Finect. Aporta datos exclusivos: **gestora, comisiones detalladas, categoría, benchmark, patrimonio, fecha de inicio** para los ~52.000 fondos/ETFs comercializados en España.

**Resolución de URL**: usa los sitemaps públicos de Finect (`funds.xml` + `etfs.xml`) para construir un índice ISIN→URL (cacheado 7 días en disco). Esto resuelve el problema de que Finect requiere un *slug* en la URL que no es deducible del ISIN.

In [ ]:
from app.services.finect_provider import FinectProvider, _get_finect_url
import requests

finect = FinectProvider()

# 4.0 Verificar resolución de URL vía sitemap
finect_url = _get_finect_url(ISIN)
if finect_url:
    print(f"✅ URL Finect resuelta: {finect_url}")
    try:
        resp = requests.get(finect_url, headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }, timeout=10)
        print(f"   HTTP {resp.status_code} | {len(resp.content)} bytes")
    except Exception as e:
        print(f"   ❌ Error: {e}")
else:
    print(f"⚠️ ISIN {ISIN} no encontrado en sitemap de Finect (no está indexado)")

# 4a. Info del fondo (header + info general + comisiones)
info_finect = finect.get_fund_info(ISIN)
print(f"\n--- Info Finect ({len(info_finect)} campos) ---")
if info_finect:
    display(pd.DataFrame(list(info_finect.items()), columns=["Campo", "Valor"]))
else:
    print("(sin datos — ISIN no disponible en Finect)")

# 4b. Asset allocation (proxy de sectores)
alloc_finect = finect.get_sector_weights(ISIN)
print("\n--- Asset Allocation Finect ---")
if alloc_finect:
    display(pd.DataFrame(list(alloc_finect.items()), columns=["Activo", "Peso (%)"]))
else:
    print("(sin datos — renderizado client-side)")

# 4c. Holdings
holdings_finect = finect.get_holdings(ISIN)
print(f"\n--- Holdings Finect ({len(holdings_finect)} posiciones) ---")
if not holdings_finect.empty:
    display(holdings_finect)
else:
    print("(sin datos — renderizado client-side)")

URL Finect: https://www.finect.com/fondos-inversion/IE00BYX5NX33
HTTP Status: 404
URL final (tras redirects): https://www.finect.com/fondos-inversion/IE00BYX5NX33
Tamaño HTML: 75695 bytes
❌ Error HTTP 404

--- Info Finect ---
(sin datos — puede que Finect no cubra este ISIN)

--- Asset Allocation Finect ---
(sin datos)

--- Holdings Finect (0 posiciones) ---
(sin datos)


## 5. CompositeProvider — Estrategia dual

El `CompositeProvider` usa **dos cadenas separadas** optimizadas para objetivos distintos:

| Operación | Cadena | Estrategia |
|---|---|---|
| `get_nav()` / `get_nav_history()` | NAV chain | Primer éxito = resultado |
| `get_fund_info()` | Data chain | Fusiona todos (primer valor gana) |
| `get_sector_weights()` / `get_country_weights()` / `get_holdings()` | Data chain | Fuente más completa gana |

In [ ]:
from app.services.data_providers import CompositeProvider
import time

composite = CompositeProvider(cache_path=cache_path)
print(f"NAV chain (⚡ velocidad):    {[type(p).__name__ for p in composite._nav_chain]}")
print(f"Data chain (📊 completitud): {[type(p).__name__ for p in composite._data_chain]}")

# 5a. NAV compuesto (usa cadena rápida)
t0 = time.time()
nav_comp = composite.get_nav(ISIN)
t_nav = (time.time() - t0) * 1000
print(f"\nNAV Composite: {nav_comp} ({t_nav:.0f}ms)")

# 5b. Info fusionada (usa cadena de datos — merge de todos)
t0 = time.time()
info_comp = composite.get_fund_info(ISIN)
t_info = (time.time() - t0) * 1000
print(f"\n--- Info Composite ({len(info_comp)} campos, {t_info:.0f}ms) ---")
display(pd.DataFrame(list(info_comp.items()), columns=["Campo", "Valor"]))

# 5c. Sectores (fuente más completa)
t0 = time.time()
sectors_comp = composite.get_sector_weights(ISIN)
t_sec = (time.time() - t0) * 1000
print(f"\n--- Sectores Composite ({len(sectors_comp)} sectores, {t_sec:.0f}ms) ---")
if sectors_comp:
    display(pd.DataFrame(list(sectors_comp.items()), columns=["Sector", "Peso (%)"]))
else:
    print("(sin datos)")

# 5d. Países/Regiones (fuente más completa)
t0 = time.time()
countries_comp = composite.get_country_weights(ISIN)
t_cou = (time.time() - t0) * 1000
print(f"\n--- Países Composite ({len(countries_comp)} regiones, {t_cou:.0f}ms) ---")
if countries_comp:
    display(pd.DataFrame(list(countries_comp.items()), columns=["País/Región", "Peso (%)"]))
else:
    print("(sin datos)")

# 5e. Holdings (fuente con más posiciones)
t0 = time.time()
holdings_comp = composite.get_holdings(ISIN)
t_hold = (time.time() - t0) * 1000
print(f"\n--- Holdings Composite ({len(holdings_comp)} posiciones, {t_hold:.0f}ms) ---")
display(holdings_comp.head(10))

Proveedores en la cadena: ['FMPProvider', 'MStarProvider', 'YFinanceProvider', 'FinectProvider']

Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BYX5NX33 (light)
NAV Composite: 13.045499801635742
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5NX33
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...


app.services.data_providers — MStarProvider: cache de IE00BYX5NX33 (detailed) contiene errores, reintentando sin cache


❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos para IE00BYX5NX33: 2053 días
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...
❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)

--- Info Composite (fusión de todos los proveedores) ---


,Campo,Valor
0,isin,IE00BYX5NX33
1,source,Morningstar
2,name,Fidelity MSCI World Index EUR P Acc
3,currency,EUR


Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5NX33
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...


...

app.services.data_providers — MStarProvider: cache de IE00BYX5NX33 (detailed) contiene errores, reintentando sin cache


❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos para IE00BYX5NX33: 2053 días
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...
❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)

--- Países Composite ---
(sin datos)
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ D

app.services.data_providers — MStarProvider: cache de IE00BYX5NX33 (detailed) contiene errores, reintentando sin cache


❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)
Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos para IE00BYX5NX33: 2053 días
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...
❌ Error general obteniendo datos para IE00BYX5NX33: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5NX33
  ⚠️ Datos con error, no se cachean para IE00BYX5NX33 (detailed)

--- Holdings Composite (0 posiciones) ---


,name,ticker,weight,market_value


## 6. Comparativa de cobertura por proveedor

Tabla resumen: ¿qué datos devuelve cada proveedor para este fondo?

In [11]:
# Construir tabla comparativa de cobertura
providers_data = {
    "FMP": {
        "NAV": nav_fmp if fmp.available else None,
        "Info campos": len(info_fmp) if fmp.available else 0,
        "Sectores": len(sectors_fmp) if fmp.available else 0,
        "Países": len(countries_fmp) if fmp.available else 0,
        "Holdings": len(holdings_fmp) if fmp.available else 0,
        "Histórico (días)": len(hist_fmp) if fmp.available else 0,
    },
    "Morningstar": {
        "NAV": nav_ms,
        "Info campos": len(info_ms),
        "Sectores": len(sectors_ms),
        "Países": len(regions_ms),
        "Holdings": len(holdings_ms),
        "Histórico (días)": len(hist_ms),
    },
    "Yahoo Finance": {
        "NAV": nav_yf,
        "Info campos": len(info_yf),
        "Sectores": 0,
        "Países": 0,
        "Holdings": 0,
        "Histórico (días)": len(hist_yf),
    },
    "Finect": {
        "NAV": None,  # Finect no provee NAV
        "Info campos": len(info_finect),
        "Sectores": len(alloc_finect),
        "Países": 0,
        "Holdings": len(holdings_finect),
        "Histórico (días)": 0,
    },
    "Composite": {
        "NAV": nav_comp,
        "Info campos": len(info_comp),
        "Sectores": len(sectors_comp),
        "Países": len(countries_comp),
        "Holdings": len(holdings_comp),
        "Histórico (días)": "—",
    },
}

df_coverage = pd.DataFrame(providers_data).T
df_coverage.index.name = "Proveedor"
print(f"=== Comparativa de cobertura para {ISIN} ===\n")
display(df_coverage)

=== Comparativa de cobertura para IE00BYX5NX33 ===



,NAV,Info campos,Sectores,Países,Holdings,Histórico (días)
Proveedor,,,,,,
FMP,NaN,0.0,0.0,0.0,0.0,0.0
Morningstar,13.0455,3.0,0.0,0.0,0.0,759.0
Yahoo Finance,13.0455,3.0,0.0,0.0,0.0,759.0
Finect,NaN,0.0,0.0,0.0,0.0,0.0
Composite,13.0455,4,0,0,0,—


### Resumen de la estrategia dual

**Para obtener el NAV** (velocidad):
1. **FMP** — 1 HTTP call, ~300ms, falla rápido (`None`) si no resuelve el ISIN
2. **YFinance** — ~500ms, buena cobertura de ISINs europeos, no requiere API key
3. **MorningStar** (light) — ~2-5s, backup que usa caché; internamente llama a yfinance

**Para obtener datos detallados** (completitud):
1. **MorningStar** (detailed) — Fuente más rica: ratings, sectores, regiones, holdings, risk score, costes. Limitación: HTTP 403 temporales
2. **FMP** — Excelente para ETFs: sector/country/holdings weightings vía API REST
3. **Finect** (sitemap + scraping) — Datos exclusivos España: gestora, comisiones detalladas, categoría, benchmark, patrimonio, fecha de inicio. 52.000 fondos indexados

**Lógica de fusión**:
- `get_fund_info()`: fusiona los 3 proveedores de datos, primer valor no-nulo gana por campo
- `get_sector_weights()` / `get_country_weights()` / `get_holdings()`: devuelve la fuente con más entradas (mayor granularidad)
- `get_nav()` / `get_nav_history()`: primer éxito en la cadena rápida

## 7. Comparativa de NAV entre proveedores

In [12]:
# Comparar precios NAV de cada fuente
nav_data = {
    "FMP": nav_fmp if fmp.available else None,
    "Morningstar": nav_ms,
    "Yahoo Finance": nav_yf,
    "Composite": nav_comp,
}

df_nav = pd.DataFrame(
    [(src, price) for src, price in nav_data.items() if price is not None],
    columns=["Fuente", "NAV (€)"],
)

if not df_nav.empty:
    mean_nav = df_nav["NAV (€)"].mean()
    df_nav["Desviación vs Media (%)"] = ((df_nav["NAV (€)"] - mean_nav) / mean_nav * 100).round(4)
    print(f"NAV medio: {mean_nav:.4f} €\n")
    display(df_nav)
else:
    print("No se obtuvo NAV de ningún proveedor.")

NAV medio: 13.0455 €



,Fuente,NAV (€),Desviación vs Media (%)
0,Morningstar,13.0455,0.0
1,Yahoo Finance,13.0455,0.0
2,Composite,13.0455,0.0


## 8. Visualización del histórico de precios

Gráfico comparando los históricos obtenidos de cada fuente para validar consistencia.

In [13]:
import plotly.graph_objects as go

fig = go.Figure()

# Histórico de cada proveedor
historics = {}
if fmp.available and not hist_fmp.empty:
    historics["FMP"] = hist_fmp
if not hist_ms.empty:
    historics["Morningstar"] = hist_ms
if not hist_yf.empty:
    historics["Yahoo Finance"] = hist_yf

colors = {"FMP": "#1f77b4", "Morningstar": "#ff7f0e", "Yahoo Finance": "#2ca02c"}

for name, df_hist in historics.items():
    fig.add_trace(go.Scatter(
        x=df_hist["date"],
        y=df_hist["price"],
        mode="lines",
        name=name,
        line=dict(color=colors.get(name, "gray")),
        opacity=0.8,
    ))

fund_name = info_comp.get("name", ISIN) if info_comp else ISIN
fig.update_layout(
    title=f"Histórico de precios — {fund_name} ({ISIN})",
    xaxis_title="Fecha",
    yaxis_title="Precio (€)",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=500,
)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 9. Comparativa de Info fields por proveedor

Tabla pivotada mostrando qué campos devuelve cada proveedor y comprobando discrepancias.

In [ ]:
# Recopilar todos los campos de info de cada proveedor
all_info = {
    "FMP": info_fmp if fmp.available else {},
    "Morningstar": info_ms,
    "Yahoo Finance": info_yf,
    "Finect": info_finect,
    "Composite": info_comp,
}

# Obtener la unión de todos los campos
all_fields = sorted(set().union(*[set(d.keys()) for d in all_info.values()]))

# Construir tabla comparativa
rows = []
for field in all_fields:
    row = {"Campo": field}
    for provider_name, info_dict in all_info.items():
        val = info_dict.get(field)
        row[provider_name] = val if val is not None else "—"
    rows.append(row)

df_fields = pd.DataFrame(rows).set_index("Campo")
print(f"Total campos únicos: {len(df_fields)}\n")
display(df_fields)

## 10. Vista unificada con `PortfolioClient.fund_details()`

Finalmente, la vista que el usuario obtendría al usar la fachada `PortfolioClient`.

In [ ]:
from app.client import PortfolioClient

# Creamos un client sin portfolio — solo usamos el proveedor de datos
client = PortfolioClient(source=None, cache_path=cache_path)

# fund_details devuelve un DataFrame con todas las métricas fusionadas
df_detail = client.fund_details(ISIN)

print(f"=== fund_details({ISIN}) — {len(df_detail)} métricas ===\n")

# Separar por categoría para mejor lectura
info_rows = df_detail[~df_detail["Metric"].str.startswith(("sector_", "country_", "holding_"))]
sector_rows = df_detail[df_detail["Metric"].str.startswith("sector_")]
country_rows = df_detail[df_detail["Metric"].str.startswith("country_")]
holding_rows = df_detail[df_detail["Metric"].str.startswith("holding_")]

print("--- Info General ---")
display(info_rows.reset_index(drop=True))

if not sector_rows.empty:
    print("\n--- Sectores ---")
    display(sector_rows.reset_index(drop=True))

if not country_rows.empty:
    print("\n--- Países/Regiones ---")
    display(country_rows.reset_index(drop=True))

if not holding_rows.empty:
    print("\n--- Top Holdings ---")
    display(holding_rows.reset_index(drop=True))